# V2 Adv GraphRAG: LLM-centric, auditable, citation-grounded
Reuses the saved index (chunks, shards, FAISS) from `Adv_graphRAG/`. **No re-embedding.** Drive must be mounted.

Flow: **parse/split -> entity+alias -> section choice -> hybrid search (30) -> cross-encoder rerank -> LLM judge (verified) -> dedupe/rank/cap -> fetch sections -> verbatim extraction (verified) -> per-drug answer -> quote + support check -> code-based abstain -> answer + citations.**

The LLM proposes, code verifies. Every step has a flag in `FLAGS` for the ablation table. Results go to `qa_results_v2.json` (same format as `qa_results_adv.json`).

In [2]:
# ===== 1. packages + Ollama (skips what is already installed) =====
import os
os.environ["JAX_PLATFORMS"] = "cpu"; os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
import subprocess, time, requests, shutil
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True)
print("pip:", sh("pip install -q ollama faiss-cpu bm25s networkx pandas numpy tqdm requests sentence-transformers rapidfuzz").returncode)
if not shutil.which("ollama"):
    sh("apt-get update -qq && apt-get install -y -qq zstd"); sh("curl -fsSL https://ollama.com/install.sh | sh")
def ollama_up():
    try: return requests.get("http://localhost:11434/api/tags", timeout=2).status_code == 200
    except Exception: return False
if not ollama_up():
    env = dict(os.environ, OLLAMA_NUM_PARALLEL="2")
    subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT, env=env)
    for _ in range(60):
        if ollama_up(): break
        time.sleep(1)
assert ollama_up(), "Ollama did not start"
for m in ["llama3:8b", "nomic-embed-text"]:
    print(m, "pull rc =", sh(f"ollama pull {m}").returncode)

pip: 0
llama3:8b pull rc = 0
nomic-embed-text pull rc = 0


In [3]:
# ===== 2. config, flags, paths =====
import re, json, pickle, glob, hashlib, html, traceback
from collections import defaultdict, Counter
import numpy as np, pandas as pd, faiss, bm25s
from tqdm.auto import tqdm

assert any(l.split()[1] == "/content/drive" for l in open("/proc/mounts")), "Drive is NOT mounted"
BASE_DIR = "/content/drive/MyDrive/Research_kgTxagent/graph_rag"
OUT_DIR  = os.path.join(BASE_DIR, "Adv_graphRAG")
GRAPH_PATH = next(p for p in ["/content/drive/MyDrive/Research_kgTxagent/Kg_dev_files_and_data/test_openfda/OpenFDA_graph.pkl",
                              os.path.join(BASE_DIR, "OpenFDA_graph.pkl")] if os.path.exists(p))
chunk_files = sorted(glob.glob(os.path.join(OUT_DIR, "chunks_*.pkl")), key=os.path.getmtime)
assert chunk_files, "no chunks_*.pkl in " + OUT_DIR
CHUNK_PATH = chunk_files[-1]
SIG = os.path.basename(CHUNK_PATH)[len("chunks_"):-len(".pkl")]
SHARD_DIR = os.path.join(OUT_DIR, f"shards_{SIG}"); FAISS_PATH = os.path.join(OUT_DIR, f"faiss_{SIG}.index")
assert os.path.isdir(SHARD_DIR), f"shard folder missing: {SHARD_DIR}"
QA_PATH = os.path.join(BASE_DIR, "QA.json")
RESULTS_PATH = os.path.join(OUT_DIR, "qa_results_v2.json")
print("SIG:", SIG)

LLM_MODEL, EMBED_MODEL = "llama3:8b", "nomic-embed-text"
EMBED_URL, GEN_URL = "http://localhost:11434/api/embed", "http://localhost:11434/api/generate"
RERANK_MODEL = "BAAI/bge-reranker-base"
RERANK_DEVICE = "cpu"                   # "cuda" is faster but competes with Llama for GPU memory

# ---- ablation switches (turn a step off to measure its effect) ----
FLAGS = {"parse": True, "alias": True, "llm_sections": True, "rerank": True, "judge": True,
         "extract": True, "per_drug": True, "quote_check": True}
# ---- settings ----
SEARCH_K, RERANK_KEEP = 60, 20          # candidates searched, kept after rerank
SPECIFIC_CAP, LIST_CAP = 3, 10          # drugs kept per sub-query (scope=specific / list)
EXTRACT_SECTIONS, SECTION_CHARS = 6, 3500
JUDGE_FALLBACK = 3                      # if the judge rejects everything, keep top-N reranked chunks (0 = abstain)
SLEEP_CALL, SLEEP_QUESTION = 1, 3       # seconds (Ollama is local, so no rate limit)
EXCLUDE_SECTIONS = {"inactive_ingredient"}
RRF_K, PRECISION_MIN = 60, 0.15
ABSTAIN_TEXT = "INSUFFICIENT FDA DATA: The retrieved label text does not contain this information."

SIG: 7c9ff259


In [4]:
# ===== 3. load graph (same lookups as before) =====
t0 = time.time()
with open(GRAPH_PATH, "rb") as f: G = pickle.load(f)
print(f"graph loaded in {time.time()-t0:.0f}s | nodes={G.number_of_nodes():,} edges={G.number_of_edges():,}")
type_counts = Counter(d.get("type") for _, d in G.nodes(data=True))
SECTION_TYPES = sorted(t for t in type_counts if t not in ("drug_generic", "drug_brand"))
generic_ids = [n for n, d in G.nodes(data=True) if d.get("type") == "drug_generic"]
brand_ids   = [n for n, d in G.nodes(data=True) if d.get("type") == "drug_brand"]
GENERIC_NAME = {n: G.nodes[n].get("name", "") for n in generic_ids}
BRAND_NAME   = {n: G.nodes[n].get("name", "") for n in brand_ids}
BRAND_TO_GENERIC, DRUG_SECTIONS = defaultdict(list), defaultdict(list)
for g in tqdm(generic_ids, desc="drug->section edges"):          # only drug->section edges (section<->section is an artifact)
    for nb in G.neighbors(g):
        t = G.nodes[nb].get("type")
        if t == "drug_brand": BRAND_TO_GENERIC[nb].append(g)
        elif t in SECTION_TYPES: DRUG_SECTIONS[g].append((nb, t))
print("generics:", len(generic_ids), "| brands:", len(brand_ids))

def section_text(sid):
    txt = G.nodes[sid].get("description") or ""
    txt = html.unescape(re.sub(r"<[^>]+>", " ", txt))
    return re.sub(r"\s+", " ", txt).strip()

graph loaded in 3s | nodes=110,843 edges=355,909


drug->section edges:   0%|          | 0/10840 [00:00<?, ?it/s]

generics: 10840 | brands: 31797


In [5]:
# ===== 4. load saved chunks + embeddings + FAISS, rebuild BM25 (about 15 s) =====
CHUNKS = pickle.load(open(CHUNK_PATH, "rb"))
shards = sorted(glob.glob(os.path.join(SHARD_DIR, "emb_*.npy")))
EMB = np.ascontiguousarray(np.vstack([np.load(p) for p in shards]), dtype="float32")
assert EMB.shape[0] == len(CHUNKS), f"embeddings {EMB.shape[0]} != chunks {len(CHUNKS)}"
faiss.normalize_L2(EMB)
INDEX = faiss.read_index(FAISS_PATH) if os.path.exists(FAISS_PATH) else None
if INDEX is None:
    INDEX = faiss.IndexFlatIP(EMB.shape[1]); INDEX.add(EMB)
print("chunks:", len(CHUNKS), "| EMB:", EMB.shape, "| FAISS:", INDEX.ntotal)

STOP = {"the","a","an","of","and","to","is","are","can","i","if","for","in","on","or","with","what","do","how","my","me","it"}
def tokenize(s):
    s = re.sub(r"(\d)([a-z])", r"\1 \2", s.lower())
    return [t for t in re.findall(r"[a-z0-9]+", s) if t not in STOP]
BM25 = bm25s.BM25(); BM25.index([tokenize(c["stamped"]) for c in CHUNKS], show_progress=False)
IDX_BY_SECTION, IDX_BY_DRUG = defaultdict(list), defaultdict(list)
for i, c in enumerate(CHUNKS):
    IDX_BY_SECTION[c["section"]].append(i); IDX_BY_DRUG[c["drug_id"]].append(i)
IDX_BY_SECTION = {k: np.array(v) for k, v in IDX_BY_SECTION.items()}
IDX_BY_DRUG = {k: np.array(v) for k, v in IDX_BY_DRUG.items()}
print("BM25 ready")

chunks: 97579 | EMB: (97579, 768) | FAISS: 97579


DEBUG:bm25s:Building index from tokens


BM25 ready


In [6]:
# ===== 5. LLM + embedding helpers, hybrid search (reused from Adv_graphRAG) =====
def _embed_call(texts):
    r = requests.post(EMBED_URL, json={"model": EMBED_MODEL, "input": texts, "truncate": True}, timeout=300)
    r.raise_for_status(); return r.json()["embeddings"]
def embed_query(q):
    return np.array(_embed_call(["search_query: " + q])[0], dtype="float32")

def llm(prompt, temperature=0.1, num_predict=500, json_mode=False, sleep=True, tries=4):
    p = {"model": LLM_MODEL, "prompt": prompt, "stream": False, "keep_alive": "30m",
         "options": {"temperature": temperature, "num_ctx": 8192, "num_predict": num_predict}}
    if json_mode: p["format"] = "json"
    out = ""
    for k in range(tries):
        try:
            r = requests.post(GEN_URL, json=p, timeout=900); r.raise_for_status()
            out = r.json()["response"].strip(); break
        except Exception as e:                                   # HTTP 500 etc: wait, retry, never kill the question
            print(f"  llm retry {k+1}/{tries}: {repr(e)[:80]}"); time.sleep(10 * (k + 1))
    if sleep and SLEEP_CALL: time.sleep(SLEEP_CALL)
    return out
def llm_json(prompt, **kw):
    raw = llm(prompt, json_mode=True, **kw)
    try: return json.loads(raw)
    except Exception:
        m = re.search(r"\{.*\}", raw, re.S)
        try: return json.loads(m.group(0)) if m else None
        except Exception: return None

_QCACHE = {}
def qvec(q):
    if q not in _QCACHE:
        v = embed_query(q); _QCACHE[q] = v / (np.linalg.norm(v) + 1e-9)
    return _QCACHE[q]
def dense_topk(qv, allowed, k):
    if allowed is None:
        _, I = INDEX.search(qv.reshape(1, -1).astype("float32"), k); return [int(i) for i in I[0] if i >= 0]
    sims = EMB[allowed] @ qv
    return [int(allowed[o]) for o in np.argsort(-sims)[:k]]
def bm25_topk(q, allowed, k):
    toks = tokenize(q)
    if not toks: return []
    scores = BM25.get_scores(toks)
    if allowed is None:
        cand = np.argpartition(-scores, min(k, len(scores) - 1))[:k]; cand = cand[np.argsort(-scores[cand])]
    else:
        cand = allowed[np.argsort(-scores[allowed])[:k]]
    return [int(c) for c in cand if scores[c] > 0]
def rrf(lists):
    out = defaultdict(float)
    for lst in lists:
        for rank, idx in enumerate(lst, start=1): out[idx] += 1.0 / (RRF_K + rank)
    return out
def hybrid_rank(queries, allowed=None, k=50):
    lists, qvs = [], []
    for q in queries:
        qv = qvec(q); qvs.append(qv)
        lists.append(dense_topk(qv, allowed, k)); lists.append(bm25_topk(q, allowed, k))
    fused = sorted(rrf(lists).items(), key=lambda x: -x[1])[:k]
    return [(i, s, max(float(EMB[i] @ qv) for qv in qvs)) for i, s in fused]

RERANKER = None
def get_reranker():
    global RERANKER
    if RERANKER is None:
        from sentence_transformers import CrossEncoder
        RERANKER = CrossEncoder(RERANK_MODEL, max_length=512, device=RERANK_DEVICE)
    return RERANKER
print("helpers ready | LLM test:", llm("Reply with the single word OK.", num_predict=5, sleep=False))

helpers ready | LLM test: OK


In [7]:
# ===== GPU check =====
import torch
print("PyTorch sees GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print(sh("nvidia-smi --query-gpu=name,utilization.gpu,memory.used --format=csv,noheader").stdout or "nvidia-smi not found: NO GPU")
llm("Say OK.", num_predict=5, sleep=False)
print(sh("ollama ps").stdout)

PyTorch sees GPU: True Tesla T4
Tesla T4, 0 %, 6579 MiB

NAME         ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL               
llama3:8b    365c0bd3c000    6.8 GB    100% GPU     8192       llamacpp    29 minutes from now    



In [8]:
# ===== 6. entity + alias detection (data-driven, no hand-written word lists) =====
def norm_tokens(s):
    s = re.sub(r"(\d)([a-z])", r"\1 \2", s.lower())
    return [t for t in re.findall(r"[a-z0-9]+", s) if t not in {"and", "the", "of", "with"}]

name_vocab = defaultdict(set)                                   # word -> drugs having it in their NAME
for g, nm in GENERIC_NAME.items():
    for t in set(norm_tokens(nm)): name_vocab[t].add(g)
for b, nm in BRAND_NAME.items():
    gens = BRAND_TO_GENERIC.get(b, [])
    for t in set(norm_tokens(nm)): name_vocab[t].update(gens)

dom_re = re.compile(r"\b([A-Za-z0-9][A-Za-z0-9\-]{2,})\.(?:com|net|org|co)\b")
tm_re  = re.compile(r"\b([A-Z][A-Za-z0-9\-]{3,})\s?(?:™|®|\(TM\)|TM\b)")
cap_re = re.compile(r"\b[A-Z][a-z]{4,}[A-Za-z\-]*\b")
text_vocab, low_df = defaultdict(set), Counter()
alias_dom, cap_hits = defaultdict(set), defaultdict(set)
squash = lambda s: re.sub(r"[^a-z0-9]", "", s.lower())
for c in tqdm(CHUNKS, desc="scanning label text"):
    d, t = c["drug_id"], c["text"]
    toks = set(norm_tokens(t))
    for w in toks:
        low_df[w] += 1
        if w in name_vocab: text_vocab[w].add(d)
    for m in dom_re.findall(t) + tm_re.findall(t): alias_dom[squash(m)].add(d)
    for m in set(cap_re.findall(t)): cap_hits[squash(m)].add(d)

# category-word filter: a name word is "specific" only if drugs that have it in the NAME also mention it in their TEXT
TOK_PREC = {}
for t, nd in name_vocab.items():
    td = text_vocab.get(t, set()); TOK_PREC[t] = (len(nd & td) / len(td)) if td else 1.0
ENT, dropped = [], []
for kind, table in (("generic", GENERIC_NAME), ("brand", BRAND_NAME)):
    for eid, nm in table.items():
        if kind == "brand" and not BRAND_TO_GENERIC.get(eid): continue
        ts = frozenset(norm_tokens(nm))
        if not ts: continue
        (ENT if max(TOK_PREC[t] for t in ts) >= PRECISION_MIN else dropped).append((ts, nm, kind, eid))
print(f"entity table kept={len(ENT):,} | dropped as category words={len(dropped):,}")

# alias table: rare product-like words found only inside label text (domains, TM marks, rare capitalised words)
ALIAS = defaultdict(set)
for a, ds in alias_dom.items():
    if len(ds) <= 5 and len(a) >= 4: ALIAS[a] |= ds
for a, ds in cap_hits.items():
    if len(ds) <= 3 and len(a) >= 5 and a not in name_vocab and low_df.get(a, 0) <= 6: ALIAS[a] |= ds
print("alias entries:", len(ALIAS))

def detect_entities(query):
    qt = set(norm_tokens(query)); hits = []
    for ts, nm, kind, eid in ENT:
        if ts <= qt or (len(ts) >= 4 and len(ts & qt) / len(ts) >= 0.8): hits.append((ts, nm, kind, eid))
    keep = [h for h in hits if not any(h[0] < o[0] for o in hits)]
    out, seen = [], set()
    for ts, nm, kind, eid in sorted(keep, key=lambda h: (-len(h[0]), len(h[1]))):
        for g in ([eid] if kind == "generic" else BRAND_TO_GENERIC.get(eid, [])):
            if g not in seen: seen.add(g); out.append({"drug_id": g, "matched": nm, "n": len(ts)})
    return out

def alias_hits(query):
    toks = [squash(t) for t in re.split(r"\s+", query)]
    cand = {t for t in toks if len(t) >= 4} | {a + b for a, b in zip(toks, toks[1:]) if a and b}
    out = []
    for a in cand:
        for g in ALIAS.get(a, ()): out.append({"drug_id": g, "matched": f"alias:{a}", "n": 99})
    return out

print("test:", [(e["matched"]) for e in detect_entities("Is Aspercreme safe for children?")][:3],
      "| alias:", [e["matched"] for e in alias_hits("Can I use Kank-a on my 1-year-old's gums?")][:3])

scanning label text:   0%|          | 0/97579 [00:00<?, ?it/s]

entity table kept=39,939 | dropped as category words=2,695
alias entries: 2692
test: ['Aspercreme'] | alias: ['alias:kanka', 'alias:kanka']


In [9]:
# ===== 7. LLM steps: parse/split + named products, section choice =====
from rapidfuzz import fuzz

def parse_question(q):
    default = {"sub_queries": [q], "named": [], "scope": "specific"}
    if not FLAGS["parse"]: return default
    p = ("Return JSON for the user question.\n"
         "1. \"named\": product or drug names the user literally wrote (brand or generic names, with strengths). "
         "Do NOT include symptoms, conditions, body parts or product categories (for example 'cold', 'antacid', 'eye drops', 'soap'). Empty list if none.\n"
         "2. \"sub_queries\": split ONLY when comparing two or more named products, or when one word has two different meanings. "
         "Split by subject, never by aspect: age limits, pregnancy, dose, warnings and so on stay together in one sub-question. "
         "Each sub-question must repeat everything asked about its subject and be self-contained. Otherwise return the question unchanged. At most 3.\n"
         "3. \"scope\": \"list\" if the user wants a list of different products, else \"specific\".\n"
         "Return JSON: {\"named\": [], \"sub_queries\": [\"...\"], \"scope\": \"specific\"}\nQuestion: " + q)
    r = llm_json(p, num_predict=250)
    try:
        sq = [s.strip() for s in r["sub_queries"] if isinstance(s, str) and s.strip()][:3]
        nm_ = [n.strip() for n in r.get("named", []) if isinstance(n, str) and n.strip()]
        sc = r.get("scope") if r.get("scope") in ("specific", "list") else "specific"
        return {"sub_queries": sq or [q], "named": nm_, "scope": sc}
    except Exception: return default

def named_in(sq, named):
    # which of the named products belong to this sub-query
    t = re.sub(r"\s+", " ", sq.lower())
    return [n for n in named if re.sub(r"\s+", " ", n.lower()) in t or fuzz.partial_ratio(n.lower(), t) >= 90]

def choose_sections(sq):
    if not FLAGS["llm_sections"]: return []
    p = ("An FDA drug label has these sections: " + ", ".join(SECTION_TYPES) + ".\n"
         "Which sections are needed to answer the question below? Pick 1 to 5 section names from that list only.\n"
         "Return JSON: {\"sections\": [\"...\"]}\nQuestion: " + sq)
    r = llm_json(p, num_predict=100)
    try: return [s for s in r["sections"] if s in SECTION_TYPES]
    except Exception: return []

In [10]:
# ===== 8. find drugs: drug mode (named products) or symptom mode (search -> rerank -> judge) =====
def norm_txt(s): return re.sub(r"\s+", " ", s.lower()).strip()
def contains(hay, needle):
    n = norm_txt(needle).strip(" .;:-•*\"'")
    return len(n) >= 8 and n in norm_txt(hay)

_LH = {}
def label_hash(d):
    if d not in _LH:
        txt = "||".join(sorted(norm_txt(section_text(sid)) for sid, t in DRUG_SECTIONS[d] if t not in EXCLUDE_SECTIONS))
        _LH[d] = hashlib.md5(txt.encode()).hexdigest()
    return _LH[d]

def dedupe_cap(drug_ids, cap):
    out, seen = [], set()
    for d in drug_ids:
        h = label_hash(d)
        if h in seen: continue
        seen.add(h); out.append(d)
        if len(out) >= cap: break
    return out

JUDGE_PROMPT = ("You judge text from an FDA drug label.\nQuestion: <<Q>>\nExcerpt (drug: <<D>>, section: <<S>>):\n<<T>>\n\n"
                "Answer relevant=true ONLY if the excerpt itself states the use or fact the question asks for. "
                "Related, similar or general text is relevant=false. If true, copy the single most useful sentence from the excerpt EXACTLY as written.\n"
                "Return JSON: {\"relevant\": true, \"sentence\": \"...\"} or {\"relevant\": false, \"sentence\": \"\"}")

def symptom_candidates(sq):
    idxA = np.concatenate([IDX_BY_SECTION[s] for s in ("indications_and_usage", "purpose") if s in IDX_BY_SECTION])
    fused = defaultdict(float)
    for i, s, c in hybrid_rank([sq], idxA, k=SEARCH_K): fused[i] += s            # indications + purpose
    for i, s, c in hybrid_rank([sq], None, k=15): fused[i] += 0.5 * s             # safety net: all sections
    cand = sorted(fused, key=lambda i: -fused[i])[:SEARCH_K]
    scores = {i: 0.0 for i in cand}
    if FLAGS["rerank"] and cand:
        sc = get_reranker().predict([(sq, CHUNKS[i]["stamped"]) for i in cand], show_progress_bar=False)
        scores = {i: float(s) for i, s in zip(cand, sc)}
        cand = sorted(cand, key=lambda i: -scores[i])[:RERANK_KEEP]
    else: cand = cand[:RERANK_KEEP]
    return cand, scores

def find_drugs(sq, scope, named, verbose=True):
    cap = LIST_CAP if scope == "list" else SPECIFIC_CAP
    hits = alias_hits(sq) if FLAGS["alias"] else []                  # alias words are rare, so safe on the whole sentence
    for n in named_in(sq, named): hits += detect_entities(n)          # drug names only from products the user named
    log = {"cands": 0, "judged_yes": 0, "judged_verified": 0}
    if hits:                                                          # DRUG MODE
        order = []
        for h in sorted(hits, key=lambda h: (-h["n"], len(h["matched"]))):
            if h["drug_id"] not in order: order.append(h["drug_id"])
        return dedupe_cap(order, cap), "drug", [h["matched"] for h in hits[:5]], log, 0.0
    cand, scores = symptom_candidates(sq)                             # SYMPTOM MODE
    log["cands"] = len(cand); kept = []
    for i in cand:
        c = CHUNKS[i]
        if not FLAGS["judge"]: kept.append(i); continue
        r = llm_json(JUDGE_PROMPT.replace("<<Q>>", sq).replace("<<D>>", c["drug"]).replace("<<S>>", c["section"]).replace("<<T>>", c["text"]), num_predict=200)
        try: rel, sent = bool(r.get("relevant")), str(r.get("sentence", ""))
        except Exception: rel, sent = False, ""
        if rel:
            log["judged_yes"] += 1
            if contains(c["text"], sent): log["judged_verified"] += 1; kept.append(i)   # code verifies the copied sentence
    if not kept and JUDGE_FALLBACK: kept = cand[:JUDGE_FALLBACK]; log["fallback"] = True
    by = defaultdict(list)
    for i in kept: by[CHUNKS[i]["drug_id"]].append(scores.get(i, 0.0))
    order = [d for d, v in sorted(by.items(), key=lambda x: -sum(sorted(x[1], reverse=True)[:2]))]   # top-2 chunk scores, not chunk count
    top = max([float(EMB[i] @ qvec(sq)) for i in kept], default=0.0)
    return dedupe_cap(order, cap), "symptom", [], log, top

In [11]:
# ===== 9. fetch sections, verbatim extraction (LLM copies, code verifies) =====
EXTRACT_PROMPT = ("Question: <<Q>>\nLabel section (<<S>>) of <<D>>:\n<<T>>\n\n"
                  "Copy, word for word, every sentence or phrase from this section that helps answer the question. One per line. "
                  "If nothing helps, reply exactly NONE. Add nothing else.")

def get_sections(d):
    out = []
    for sid, t in DRUG_SECTIONS[d]:
        if t in EXCLUDE_SECTIONS: continue
        tx = section_text(sid)
        if tx: out.append((sid, t, tx))
    return out

def pick_sections(d, sq, chosen):
    secs = get_sections(d)
    first = [s for s in secs if s[1] in chosen]; rest = [s for s in secs if s[1] not in chosen]
    if FLAGS["rerank"] and rest:
        sc = get_reranker().predict([(sq, s[2][:1500]) for s in rest], show_progress_bar=False)
        rest = [s for _, s in sorted(zip(sc, rest), key=lambda x: -x[0])]
    return first + rest[:max(0, EXTRACT_SECTIONS - len(first))]

def extract_for_drug(d, sq, chosen, sources):
    snips = []
    for sid, stype, tx in pick_sections(d, sq, chosen):
        tx = tx[:SECTION_CHARS]
        if not FLAGS["extract"]: lines = [tx[:700]]
        else:
            raw = llm(EXTRACT_PROMPT.replace("<<Q>>", sq).replace("<<S>>", stype).replace("<<D>>", GENERIC_NAME[d]).replace("<<T>>", tx), num_predict=300)
            lines = [] if raw.strip().upper().startswith("NONE") else [re.sub(r"^[\-\*•\d\.\)\s]+", "", l).strip() for l in raw.splitlines()]
        seen = set()
        for l in lines:
            if len(l) < 8 or l in seen or not (contains(tx, l) or not FLAGS["extract"]): continue   # keep only verified copies
            seen.add(l)
            s = {"tag": f"S{len(sources)+1}", "drug_id": d, "drug": GENERIC_NAME[d], "section": stype, "node": sid, "text": l, "sq": sq}
            sources.append(s); snips.append(s)
    return snips

In [12]:
# ===== 10. per-drug answer, quote + support check, merge, code-based abstain =====
GEN_PROMPT = ("Answer the question using ONLY the numbered label snippets below. Rules: each claim is one full sentence that states a fact and names the drug; "
              "a drug name alone is not a claim. For each claim give at most 3 source tags and copy the exact words from the snippet that support it. "
              "If the snippets do not answer the question, return an empty claims list.\n"
              "Return JSON: {\"claims\": [{\"claim\": \"...\", \"sources\": [\"S1\"], \"quote\": \"...\"}]}\n"
              "Question: <<Q>>\nSnippets:\n<<SN>>")

def generate_claims(question, snips):
    if not snips: return []
    sn = "\n".join(f"[{s['tag']}] ({s['drug']} | {s['section']}) {s['text']}" for s in snips[:15])   # cap prompt size
    r = llm_json(GEN_PROMPT.replace("<<Q>>", question).replace("<<SN>>", sn), num_predict=600)
    try: cl = [c for c in r["claims"] if isinstance(c, dict) and isinstance(c.get("claim"), str)]
    except Exception: return []
    return [c for c in cl if len(c["claim"].split()) >= 5 and len(c.get("sources") or []) <= 4]   # drop name-only / over-cited claims

def check_claims(claims, sources):
    by = {s["tag"]: s for s in sources}
    for c in claims:
        tags = [t for t in (c.get("sources") or []) if isinstance(t, str)]
        c["tags"] = tags
        if not tags: c["status"] = "no_citation"; continue
        if any(t not in by for t in tags): c["status"] = "invalid_citation"; continue
        ev = "\n".join(by[t]["text"] for t in tags)
        if FLAGS["quote_check"] and not contains(ev, str(c.get("quote", ""))): c["status"] = "quote_not_in_source"; continue
        v = llm(f"Label text:\n{ev}\n\nClaim: {c['claim']}\n\nDoes the label text directly support the claim? Answer with one word: YES or NO.",
                temperature=0.0, num_predict=3)
        c["status"] = "supported" if v.upper().startswith("YES") else "unsupported"
    return claims

def answer_question(question, verbose=True):
    t0 = time.time()
    if verbose: print("\nQUESTION:", question)
    parsed = parse_question(question)
    sources, rows, modes, ents, logs, tops, extracted = [], [], [], [], [], [], {}
    for sq in parsed["sub_queries"]:
        chosen = choose_sections(sq)
        drugs, mode, e, log, top = find_drugs(sq, parsed["scope"], parsed["named"], verbose)
        modes.append(mode); ents += e; logs.append(log); tops.append(top)
        if verbose: print(f"  sub-query: {sq!r} | mode={mode} | scope={parsed['scope']} | named={parsed['named']} | sections={chosen} | drugs={[GENERIC_NAME[d] for d in drugs]} | judge={log}")
        for d in drugs:
            if d not in extracted: extracted[d] = extract_for_drug(d, sq, chosen, sources)   # each drug is extracted once
            rows.append({"drug_id": d, "drug": GENERIC_NAME[d], "sq": sq, "snips": extracted[d]})
    claims, reason = [], None
    if not rows: reason = "no drugs found"
    elif not sources: reason = "no verified evidence"
    else:
        by_drug = {}
        for r in rows: by_drug.setdefault(r["drug_id"], r)
        groups = [(r["snips"], r["sq"]) for r in by_drug.values()] if FLAGS["per_drug"] else [(sources, question)]
        for snips, sq in groups: claims += generate_claims(sq, snips)
        claims = check_claims(claims, sources)
        if not any(c["status"] == "supported" for c in claims): reason = "no supported claims"
    final = ABSTAIN_TEXT if reason else "\n".join(("- " if c["status"] == "supported" else f"- [FLAGGED: {c['status']}] ") + c["claim"] + " " + "".join(f"[{t}]" for t in c["tags"]) for c in claims)
    rec = {"question": question, "sub_queries": parsed["sub_queries"], "named": parsed["named"], "scope": parsed["scope"], "mode": "+".join(sorted(set(modes))),
           "entities": ents, "intent": [], "drugs": [r["drug"] for r in rows], "sections_in_context": sorted({s["section"] for s in sources}),
           "sources": [{k: v for k, v in s.items() if k not in ("drug_id", "sq")} | {"cos": None} for s in sources], "top_cos": round(max(tops, default=0.0), 3),
           "context_chars": sum(len(s["text"]) for s in sources), "judge": logs, "claims": claims,
           "unsupported_rate": round(sum(c["status"] != "supported" for c in claims) / len(claims), 3) if claims else None,
           "raw_answer": final, "abstained": bool(reason), "abstain_reason": reason, "final_answer": final, "seconds": round(time.time() - t0, 1)}
    if verbose: print("  ANSWER:\n", final, f"\n  done in {rec['seconds']}s | abstain={rec['abstain_reason']}")
    return rec

In [13]:
# ===== 11. quick demo (3 questions) =====
for q in ["What should I avoid if I am pregnant and using lidocaine?",
          "Can I use Kank-a on my 1-year-old's gums for teething?",
          "Compare the indications and purpose of Acetaminophen 500 mg with Caffeine 65 mg versus Acetaminophen 500 mg."]:
    answer_question(q); print("-" * 80)


QUESTION: What should I avoid if I am pregnant and using lidocaine?
  sub-query: 'What should I avoid if I am pregnant and using lidocaine?' | mode=drug | scope=specific | named=['lidocaine'] | sections=['indications_and_usage', 'pregnancy_or_breast_feeding'] | drugs=['LIDOCAINE', 'LIDOCAINE HYDROCHLORIDE', 'LIDOCAINE 4%'] | judge={'cands': 0, 'judged_yes': 0, 'judged_verified': 0}


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

  ANSWER:
 - If you are pregnant and using lidocaine, you should avoid using it in large quantities. [S4]
- You should also avoid using lidocaine over raw surfaces or blistered areas if you are pregnant. [S5]
- If you are pregnant and using lidocaine, you should avoid traumatizing the mucosa in the area of application. [S7]
- If you are pregnant and using lidocaine, you should avoid using it on wounds or damaged, broken, or irritated skin. [S13]
- If you are pregnant and using lidocaine, you should avoid using it if you have had an allergic reaction to lidocaine or other local anesthetics. [S12]
- If you are pregnant and using lidocaine, you should avoid using it if you have not had an allergic reaction to lidocaine or other local anesthetics. [S11] 
  done in 116.0s | abstain=None
--------------------------------------------------------------------------------

QUESTION: Can I use Kank-a on my 1-year-old's gums for teething?
  sub-query: "Can I use Kank-a on my 1-year-old's gums for t

In [14]:
# ===== 12. run the QA set (resumable, saved after every question) =====
RESET = True      # True = back up the old results file and start fresh. Set to False when resuming after a kernel restart.
if RESET and "RESET_DONE" not in globals() and os.path.exists(RESULTS_PATH):
    bak = RESULTS_PATH.replace(".json", f"_run1_{int(time.time())}.json"); os.rename(RESULTS_PATH, bak); print("old results backed up to", bak)
RESET_DONE = True

assert os.path.exists(QA_PATH), f"QA.json not found at {QA_PATH}"
QA = json.load(open(QA_PATH))
RESULTS = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}
todo = [q for q in QA if q not in RESULTS]
print(f"QA questions: {len(QA)} | done: {len(RESULTS)} | to run: {len(todo)}")

nm = lambda s: re.sub(r"[^A-Z0-9]+", " ", s.upper()).strip()
SKIP_WORDS = {"and", "the", "with", "mg", "unspecified", "form", "tablet", "tablets", "cream", "ointment", "gel", "solution", "topical", "oral", "liquid"}
ing = lambda s: frozenset(t for t in re.findall(r"[a-z]{3,}", s.lower()) if t not in SKIP_WORDS)   # ingredient set of a drug name
gold_drugs = lambda qa: [m for t in qa["trace"] for m in re.findall(r"DRUG: (.*?) \| SOURCE", t)]
failed = []
for n, qid in enumerate(todo, 1):
    qa = QA[qid]; print(f"\n######## {qid} ({n}/{len(todo)})")
    try: rec = answer_question(qa["question"], verbose=True)
    except Exception as e:
        print("!! ERROR", qid, repr(e)); traceback.print_exc(); failed.append(qid); continue
    gold_raw = gold_drugs(qa)
    gold, got = [nm(g) for g in gold_raw], [nm(d) for d in rec["drugs"]]
    got_ing = {ing(d) for d in rec["drugs"]}
    rec["gold_drugs"] = gold
    rec["drug_recall"] = round(sum(g in got for g in gold) / len(gold), 2) if gold else None                 # exact name
    rec["drug_recall_ing"] = round(sum(ing(g) in got_ing for g in gold_raw) / len(gold_raw), 2) if gold_raw else None   # same ingredients
    rec["section_recall"] = round(len(set(qa["sections"]) & set(rec["sections_in_context"])) / len(qa["sections"]), 2) if qa["sections"] else None
    rec["answerable"], rec["answer_key"] = qa["answerable"], qa["answer"]
    rec["abstain_correct"] = (rec["abstained"] == (not qa["answerable"]))
    RESULTS[qid] = rec
    json.dump(RESULTS, open(RESULTS_PATH, "w"), indent=2)
    print(f"  >> gold={gold} | drug_recall={rec['drug_recall']} (ingredient match {rec['drug_recall_ing']}) section_recall={rec['section_recall']} abstain_correct={rec['abstain_correct']}")
    time.sleep(SLEEP_QUESTION)
print("\nSaved", len(RESULTS), "results to", RESULTS_PATH)
if failed: print("Errored (re-run this cell to retry):", failed)

old results backed up to /content/drive/MyDrive/Research_kgTxagent/graph_rag/Adv_graphRAG/qa_results_v2_run1_1794628572.json
QA questions: 60 | done: 0 | to run: 60

######## T1-001 (1/60)

QUESTION: What should I avoid if I am pregnant and using lidocaine?
  sub-query: 'What should I avoid if I am pregnant and using lidocaine?' | mode=drug | scope=specific | named=['lidocaine'] | sections=['indications_and_usage', 'pregnancy_or_breast_feeding'] | drugs=['LIDOCAINE', 'LIDOCAINE HYDROCHLORIDE', 'LIDOCAINE 4%'] | judge={'cands': 0, 'judged_yes': 0, 'judged_verified': 0}
  ANSWER:
 - If you are pregnant and using lidocaine, you should avoid using it in large quantities. [S4]
- You should also avoid using lidocaine over raw surfaces or blistered areas if you are pregnant. [S5]
- You should avoid using lidocaine in patients less than 3 years of age unless safer alternatives are not available or have been tried but failed. [S8]
- If you are pregnant and using lidocaine, you should avoid usin

In [15]:
# ===== 13. summary by test type =====
df = pd.DataFrame([{"qid": k, "type": k.split("-")[0], **{x: v.get(x) for x in
      ["drug_recall", "drug_recall_ing", "section_recall", "abstained", "abstain_correct", "unsupported_rate", "seconds"]}} for k, v in RESULTS.items()])
print(df.groupby("type").agg(n=("qid", "count"), drug_recall=("drug_recall", "mean"), drug_recall_ing=("drug_recall_ing", "mean"),
      section_recall=("section_recall", "mean"), abstain_correct=("abstain_correct", "mean"),
      unsupported_rate=("unsupported_rate", "mean"), avg_seconds=("seconds", "mean")).round(2))
print("\nOverall drug_recall:", round(df.drug_recall.mean(), 2), "| ingredient match:", round(df.drug_recall_ing.mean(), 2),
      "| section_recall:", round(df.section_recall.mean(), 2), "| abstain_correct:", round(df.abstain_correct.mean(), 2),
      "| unsupported_rate:", round(df.unsupported_rate.mean(), 2))
print("\nGold drug not retrieved (ingredient match):", list(df[df.drug_recall_ing < 1].qid))

      n  drug_recall  drug_recall_ing  section_recall  abstain_correct  \
type                                                                     
T1    6         0.33             0.33            0.84             1.00   
T2    8         0.00             0.00            0.75             1.00   
T3    9         0.67             0.78            0.96             0.78   
T4    9         0.22             0.28            1.00             1.00   
T5    9         0.50             0.61            1.00             0.78   
T6    9         1.00             1.00             NaN             0.89   
T7    5         0.40             0.60            1.00             1.00   
T8    5         0.30             0.30            0.92             0.80   

      unsupported_rate  avg_seconds  
type                                 
T1                0.14        98.82  
T2                0.23       258.60  
T3                0.22        79.59  
T4                0.21       351.37  
T5                0.47        8

In [16]:
# ===== 14. T8 citation-integrity test: inject a made-up claim, the check must flag it =====
TEST_QID = "T3-005"
rec = RESULTS.get(TEST_QID) or answer_question(QA[TEST_QID]["question"], verbose=False)
src = [dict(s) for s in rec["sources"]]
fake = {"claim": "The ointment is safe for children over 6 years of age.", "sources": [src[0]["tag"]], "quote": "safe for children over 6 years of age"}
real = [c for c in rec["claims"] if c.get("status") == "supported"][:1]
checked = check_claims([dict(c) for c in real] + [fake], src)
for c in checked: print(c["status"].upper(), "|", c["claim"])
print("\nPASS" if checked[-1]["status"] != "supported" else "\nFAIL: injected claim was accepted")

SUPPORTED | If Xiclofen ointment burns, stop using it and consult a doctor.
QUOTE_NOT_IN_SOURCE | The ointment is safe for children over 6 years of age.

PASS
